# External model — streaming completion

OpenAI-compatible **`POST …/v1/chat/completions`** on the gateway host with **`stream: true`**. Response is read as **SSE** and printed as tokens arrive.

**Prereq:** Python 3.9+ (stdlib: `json`, `ssl`, `urllib`).

**Credentials:** Same pattern as **`maas-validation-demo-with-key.ipynb`**: `MAAS_API_KEY` / `API_KEY` / **Demo quick swap** for the bearer token (mint with `POST …/maas-api/v1/api-keys` and subscription **`external-models-subscription`** as in your shell script). **`VERIFY_TLS`** from the environment (`1` / `true` for strict TLS).

**Two presets:** **Box A** = GPT-4o, **Box B** = Claude. Set **`ACTIVE_EXTERNAL`** to **`"gpt4o"`** or **`"claude"`** to swap.

## Demo quick swap

Optional paste between takes. Run this cell, then **External models (pick one)**.

| Variable | Effect |
|----------|--------|
| `DEMO_API_KEY` | Non-empty → overrides `MAAS_API_KEY` / `API_KEY` env. |
| `DEMO_EXTERNAL_MODEL_NAME` | Non-empty → overrides the resolved model id. |
| `DEMO_EXTERNAL_CHAT_COMPLETIONS_URL` | Non-empty → overrides the resolved chat URL. |

In [ ]:
DEMO_API_KEY = ""
DEMO_EXTERNAL_MODEL_NAME = ""
DEMO_EXTERNAL_CHAT_COMPLETIONS_URL = ""

## External models (pick one)

Defaults match the gateway routes below. Change **`ACTIVE_EXTERNAL`** to **`"gpt4o"`** or **`"claude"`** to swap. Edit **`GW_HOST`** if your cluster route differs.

In [ ]:
import json
import os
import ssl
import urllib.error
import urllib.request

# Gateway host (same idea as GW_HOST=maas.apps.... in curl)
GW_HOST = "maas.apps.rosa.jland.li5b.p3.openshiftapps.com"

# --- Box A: GPT-4o ---
_PRESET_GPT4O = {
    "url": f"https://{GW_HOST}/gpt-4o/v1/chat/completions",
    "model": "gpt-4o",
}

# --- Box B: Claude ---
_PRESET_CLAUDE = {
    "url": f"https://{GW_HOST}/claude-sonnet-4-20250514/v1/chat/completions",
    "model": "claude-sonnet-4-20250514",
}

# Swap between the two: "gpt4o" or "claude"
ACTIVE_EXTERNAL = "gpt4o"

_presets = {"gpt4o": _PRESET_GPT4O, "claude": _PRESET_CLAUDE}
if ACTIVE_EXTERNAL not in _presets:
    raise SystemExit('ACTIVE_EXTERNAL must be "gpt4o" or "claude"')
_sel = _presets[ACTIVE_EXTERNAL]
EXTERNAL_CHAT_COMPLETIONS_URL = _sel["url"]
EXTERNAL_MODEL_NAME = _sel["model"]

_dm = globals().get("DEMO_EXTERNAL_MODEL_NAME", "")
if isinstance(_dm, str) and _dm.strip():
    EXTERNAL_MODEL_NAME = _dm.strip()
_du = globals().get("DEMO_EXTERNAL_CHAT_COMPLETIONS_URL", "")
if isinstance(_du, str) and _du.strip():
    EXTERNAL_CHAT_COMPLETIONS_URL = _du.strip()

_ak = globals().get("DEMO_API_KEY", "")
if isinstance(_ak, str) and _ak.strip():
    API_KEY = _ak.strip()
else:
    API_KEY = (os.environ.get("MAAS_API_KEY") or os.environ.get("API_KEY") or "").strip()

VERIFY_TLS = os.environ.get("VERIFY_TLS", "").lower() in ("1", "true", "yes")

if not API_KEY:
    raise SystemExit("Set DEMO_API_KEY or MAAS_API_KEY / API_KEY.")

print("ACTIVE_EXTERNAL :", ACTIVE_EXTERNAL)
print("Model id          :", EXTERNAL_MODEL_NAME)
print("POST URL          :", EXTERNAL_CHAT_COMPLETIONS_URL)
print("VERIFY_TLS        :", VERIFY_TLS)
print("API key set       :", bool(API_KEY))

## Stream one chat completion

Sends **`messages`**, **`stream: true`**, reads **`text/event-stream`** (SSE), prints **`choices[].delta.content`** chunks, then a short summary.

In [ ]:
USER_MESSAGE = "Say hello in one short sentence."
MAX_TOKENS = 256


def _ssl_ctx():
    ctx = ssl.create_default_context()
    if not VERIFY_TLS:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
    return ctx


def stream_chat_completions(
    url: str,
    *,
    api_key: str,
    model: str,
    user_message: str,
    max_tokens: int,
) -> None:
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": user_message}],
        "max_tokens": max_tokens,
        "stream": True,
    }
    body = json.dumps(payload).encode("utf-8")
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "Accept": "text/event-stream",
    }
    req = urllib.request.Request(url, data=body, headers=headers, method="POST")
    ctx = _ssl_ctx()
    total_chars = 0
    last_usage = None
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=300) as resp:
            while True:
                line = resp.readline()
                if not line:
                    break
                s = line.decode("utf-8", errors="replace").strip()
                if not s or s.startswith(":"):
                    continue
                if not s.startswith("data: "):
                    continue
                data = s[6:].strip()
                if data == "[DONE]":
                    break
                try:
                    obj = json.loads(data)
                except json.JSONDecodeError:
                    continue
                if isinstance(obj, dict) and "usage" in obj:
                    last_usage = obj.get("usage")
                for ch in obj.get("choices") or []:
                    if not isinstance(ch, dict):
                        continue
                    delta = ch.get("delta")
                    if isinstance(delta, dict):
                        c = delta.get("content")
                        if c:
                            print(c, end="", flush=True)
                            total_chars += len(c)
    except urllib.error.HTTPError as e:
        err = e.read().decode("utf-8", errors="replace")
        print("HTTP", e.code, err[:2000])
        return
    print()
    print("---")
    print("Stream finished. Approx printed chars:", total_chars)
    if last_usage:
        print("usage:", last_usage)


stream_chat_completions(
    EXTERNAL_CHAT_COMPLETIONS_URL,
    api_key=API_KEY,
    model=EXTERNAL_MODEL_NAME,
    user_message=USER_MESSAGE,
    max_tokens=MAX_TOKENS,
)